In [23]:
import pandas as pd 
from torchvision.datasets import MNIST
import torchvision.transforms as tfms
from torch.utils.data import DataLoader
import torch 
import torch.nn as nn 
import torch.optim as optim

In [13]:
tfm = tfms.Compose([
    tfms.ToTensor(),
    tfms.Normalize((0.5,),(0.5,))
]) 

trainset = MNIST(root = "./MNIST" , train = True , download = True ,transform = tfm)
testset = MNIST(root = "./MNIST" , train = False , download = True ,transform = tfm)

train_loader = DataLoader(trainset ,batch_size = 64, shuffle = True )
test_loader = DataLoader(testset ,batch_size = 64, shuffle = True )

In [33]:
class RNN(nn.Module):
    def __init__(self , input_size , num_layers = 1  , hidden_size = 128 ,num_classes = 10):
        super(RNN,self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # rnn layer
        self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first=True)

        # fully connected layer 
        self.fc = nn.Linear(hidden_size , num_classes)
    def forward(self , x ) :
        ho = torch.zeros(self.num_layers , x.size(0) , self.hidden_size).to(x.device)
        out ,_ = self.rnn (x,ho)
        output = self.fc(out[:,-1,:])
        return output

In [34]:
model = RNN(input_size = 28)

criterion = nn.CrossEntropyLoss() # binary cross entropy
optimizer = optim.Adam(model.parameters())

In [38]:
epochs = 20 
for epoch in range(epochs) :
    model.train()
    epoch_loss = 0
    for xb , yb in train_loader :
        optimizer.zero_grad()
        
        xb = xb.squeeze(1) 
        
        output = model(xb)  
        
        loss = criterion(output,yb)
        
        loss.backward()
        optimizer.step()
        epoch_loss = epoch_loss + loss.item()
    print(f"{epoch+1}/10 epoch_loss is :{epoch_loss/len(train_loader)} ")

1/10 epoch_loss is :0.12417135770811932 
2/10 epoch_loss is :0.12554911786972334 
3/10 epoch_loss is :0.12586921942718168 
4/10 epoch_loss is :0.11145854144861528 
5/10 epoch_loss is :0.12166095242700549 
6/10 epoch_loss is :0.11765094165569112 
7/10 epoch_loss is :0.11513355116683568 
8/10 epoch_loss is :0.10903071373331744 
9/10 epoch_loss is :0.11869915639078106 
10/10 epoch_loss is :0.11985129542819568 
11/10 epoch_loss is :0.11452464305801686 
12/10 epoch_loss is :0.11026659210883716 
13/10 epoch_loss is :0.11194703323063629 
14/10 epoch_loss is :0.1139115592770612 
15/10 epoch_loss is :0.11912918347579393 
16/10 epoch_loss is :0.11667899449250654 
17/10 epoch_loss is :0.10245504487466726 
18/10 epoch_loss is :0.11596838241693244 
19/10 epoch_loss is :0.11070174967417362 
20/10 epoch_loss is :0.10616992245505709 


In [40]:
# evaluation 
total = 0.0
correct_pred = 0.0
with torch.no_grad ():
    model.eval()
    for images , labels in test_loader :
        images = images.squeeze(1)
        output = model(images)
        _ ,pred= torch.max(output ,1)
        
        correct_pred += (pred == labels).sum().item()
        total += labels.size(0)

print(f"accuracy = {((correct_pred)/(total))*100}% ") 


accuracy = 96.78999999999999% 
